### Cell 2 sets up the environment and authentication. It reads RUN_DIR from environment variables (no hardcoding), uses the locked GIS auth pattern with profile='josephzr', validates authentication using gis.users.me (not gis.properties.user), and creates a curated DataFrame of 8 Australian mine sites with relevant metadata. Ends with print confirmation.

In [ ]:
# ── Cell 2: Environment and Auth ──────────────────────────────────────────────
# Set up run directory from environment, authenticate to ArcGIS Online,
# and create curated Australian mines reference data

# Get RUN_DIR from environment variable
RUN_DIR = os.environ.get("RUN_DIR")
if RUN_DIR is None:
    raise EnvironmentError("RUN_DIR environment variable is not set")

print(f"RUN_DIR: {RUN_DIR}")

# Authenticate using locked profile pattern
gis = GIS(profile="josephzr")

# Validate authentication using gis.users.me (not gis.properties.user)
user = gis.users.me
print(f"Authenticated as: {user.username}")

# Create curated DataFrame of 8 Australian mine sites
australian_mines = pd.DataFrame({
    "mine_name": [
        "Olympic Dam",
        "Super Pit (Kalgoorlie)",
        "Mount Isa",
        "Argyle Diamond Mine",
        "Boddington Gold Mine",
        "Prominent Hill",
        "Pilbara Iron Ore",
        "Hunter Valley Coal"
    ],
    "commodity": [
        "Copper, Uranium, Gold",
        "Gold",
        "Copper, Lead, Zinc",
        "Diamonds",
        "Gold, Copper",
        "Copper, Gold",
        "Iron Ore",
        "Coal"
    ],
    "state": [
        "South Australia",
        "Western Australia",
        "Queensland",
        "Western Australia",
        "Western Australia",
        "South Australia",
        "Western Australia",
        "New South Wales"
    ],
    "latitude": [
        -30.4500,
        -30.7489,
        -20.7256,
        -16.7114,
        -32.7475,
        -29.7167,
        -22.3000,
        -32.3833
    ],
    "longitude": [
        136.8833,
        121.5031,
        139.4927,
        128.3903,
        116.8653,
        135.5333,
        118.5000,
        150.8833
    ],
    "status": [
        "Operating",
        "Operating",
        "Operating",
        "Closed (2020)",
        "Operating",
        "Operating",
        "Operating",
        "Operating"
    ]
})

print(f"Curated mines DataFrame: {len(australian_mines)} Australian sites")
print(australian_mines[["mine_name", "state", "commodity"]].to_string(index=False))
print("\nEnvironment and Auth OK")

### Cell 3 searches ArcGIS Online for public Australian mining layers using outside_org=True to search public content. It validates results geographically by sampling up to 20 points from each candidate layer and checking that at least 60% fall within the Australia bounding box (lat -45 to -10, lon 113 to 154). Once a valid layer is found, it queries the full layer to a Spatially-enabled DataFrame (SDF). Uses gis.users.me pattern and ends with print confirmation.

In [ ]:
# ── Cell 3: ArcGIS Data Pull ──────────────────────────────────────────────────
# Search for public Australian mining layers, validate geographically,
# and query valid layer to SDF

# Australia bounding box for geographic validation
AUS_LAT_MIN, AUS_LAT_MAX = -45, -10
AUS_LON_MIN, AUS_LON_MAX = 113, 154

def is_in_australia(lat, lon):
    """Check if coordinates fall within Australia bounding box."""
    return (AUS_LAT_MIN <= lat <= AUS_LAT_MAX and 
            AUS_LON_MIN <= lon <= AUS_LON_MAX)

def validate_layer_geography(layer, sample_size=20, threshold=0.6):
    """
    Validate a layer by sampling points and checking if >= threshold
    fall within Australia bounding box.
    """
    try:
        # Query a sample of features
        result = layer.query(where="1=1", result_record_count=sample_size, out_sr=4326)
        features = result.features
        
        if not features:
            return False, 0, 0
        
        in_aus_count = 0
        total_valid = 0
        
        for feature in features:
            geom = feature.geometry
            if geom is None:
                continue
            
            # Handle different geometry types - extract centroid/point
            if 'x' in geom and 'y' in geom:
                lon, lat = geom['x'], geom['y']
            elif 'rings' in geom:  # Polygon - use centroid approximation
                ring = geom['rings'][0]
                lon = sum(p[0] for p in ring) / len(ring)
                lat = sum(p[1] for p in ring) / len(ring)
            elif 'paths' in geom:  # Polyline - use midpoint
                path = geom['paths'][0]
                mid_idx = len(path) // 2
                lon, lat = path[mid_idx][0], path[mid_idx][1]
            else:
                continue
            
            total_valid += 1
            if is_in_australia(lat, lon):
                in_aus_count += 1
        
        if total_valid == 0:
            return False, 0, 0
        
        ratio = in_aus_count / total_valid
        return ratio >= threshold, in_aus_count, total_valid
        
    except Exception as e:
        print(f"    Validation error: {e}")
        return False, 0, 0

# Search for public Australian mining layers
print("Searching ArcGIS Online for public Australian mining layers...")
search_results = gis.content.search(
    query="australia mining mines",
    item_type="Feature Layer",
    outside_org=True,
    max_items=15
)

print(f"Found {len(search_results)} candidate items")

# Validate each result geographically
valid_layer = None
valid_item = None

for item in search_results:
    print(f"\nChecking: {item.title}")
    print(f"  Owner: {item.owner}, Views: {item.numViews}")
    
    try:
        # Get the feature layer(s) from the item
        layers = item.layers if hasattr(item, 'layers') else []
        
        if not layers:
            print("  No layers found, skipping...")
            continue
        
        for idx, layer in enumerate(layers):
            print(f"  Layer {idx}: {layer.properties.name if hasattr(layer.properties, 'name') else 'unnamed'}")
            
            is_valid, in_aus, total = validate_layer_geography(layer)
            
            if total > 0:
                print(f"    Geographic validation: {in_aus}/{total} points in Australia ({in_aus/total*100:.1f}%)")
            
            if is_valid:
                print("    ✓ VALID - meets 60% threshold")
                valid_layer = layer
                valid_item = item
                break
            else:
                print("    ✗ Does not meet geographic threshold")
        
        if valid_layer:
            break
            
    except Exception as e:
        print(f"  Error accessing item: {e}")
        continue

# Query valid layer to SDF
if valid_layer is not None:
    print(f"\n{'='*60}")
    print(f"Querying valid layer: {valid_item.title}")
    
    # Query all features to Spatially-enabled DataFrame
    mining_sdf = valid_layer.query(where="1=1", out_sr=4326).sdf
    
    print(f"Retrieved {len(mining_sdf)} features")
    print(f"Columns: {list(mining_sdf.columns)}")
    print(f"\nFirst 5 rows preview:")
    print(mining_sdf.head())
else:
    print("\nNo valid Australian mining layer found in search results")
    mining_sdf = pd.DataFrame()

print("\nArcGIS Data Pull OK")